imports

In [5]:
import os
import sys
import matplotlib as mpl

sys.path.append("..")
mpl.rcParams["animation.embed_limit"] = 100

from cleanroom_cfd.config import SimulationConfig
from cleanroom_cfd.setup import (
    build_room_setup,
    build_initial_fields,
)
from cleanroom_cfd.data_pipeline import (
    load_timeline_window,
    build_machine_state,
    build_human_entities_for_time,
    update_machine_entities_for_time,
)
from cleanroom_cfd.simulation import compute_time_step, compute_tau_field, cfd_step
from cleanroom_cfd.analysis import run_simulation_and_collect, save_metric_plots
from cleanroom_cfd.visualization import animate_simulation

experiment settings


In [6]:
cfg = SimulationConfig()

csv_path = "../assets/object_timeline_temperature.csv"
svg_path = "../assets/Feynmann room_inked_good.svg"
results_dir = "../results"

table_window_minutes = 60
save_interval_s = 30.0

cfg.res = 10
cfg.sock_speed = 0.45
cfg.target_time = table_window_minutes * 60
cfg.frames = int(cfg.target_time / save_interval_s)

os.makedirs(results_dir, exist_ok=True)

load timeline data

In [7]:
timeline_data = load_timeline_window(
    csv_path=csv_path,
    table_window_minutes=table_window_minutes
)

print("Window start:", timeline_data["first_ts"])
print("Window end:", timeline_data["end_ts"])
print("Rows in window:", len(timeline_data["df_window"]))
print("Unique timestamps in window:", len(timeline_data["unique_times"]))
print("Labels in window:", sorted(timeline_data["df_window"]["canonical_label"].dropna().unique().tolist()))
print("people_or_machine values:", sorted(timeline_data["df_window"]["people_or_machine"].dropna().unique().tolist()))

Window start: 2025-04-23 13:28:31
Window end: 2025-04-23 14:28:31
Rows in window: 1146
Unique timestamps in window: 120
Labels in window: ['Cableduct', 'Chair', 'Machine', 'Person', 'Screen', 'Window']
people_or_machine values: ['machine', 'person']


geometry and room setup

In [8]:
room = build_room_setup(cfg, svg_path)

print("Room size:", room["svg_width_m"], "x", room["svg_height_m"])
print("Grid:", room["grid_w"], "x", room["grid_h"])
print("Sock rows:", room["src_y0"], room["src_y1"])

Room size: 15.0 x 12.0
Grid: 150 x 120
Sock rows: 78 82


build static entities and machine state

In [9]:
base_entities_list = [
    # Middle passthrough furniture
    room["entity_factory"]("furniture_passthrough", x_m=2.0, y_m=3.8, width_m=8.8, height_m=2.3, id_name="middle_1"),
    room["entity_factory"]("furniture_passthrough", x_m=10.8, y_m=3.0, width_m=1.0, height_m=3.1, id_name="middle_2"),

    # Existing passthrough furniture
    room["entity_factory"]("furniture_passthrough", x_m=1.8, y_m=0.0, width_m=3.6, height_m=1.0, id_name="table_1"),
    room["entity_factory"]("furniture_passthrough", x_m=5.7, y_m=0.0, width_m=3.9, height_m=1.0, id_name="table_2"),
    room["entity_factory"]("furniture_passthrough", x_m=9.8, y_m=0.0, width_m=3.8, height_m=1.0, id_name="table_3"),
]

machine_state = build_machine_state(
    df_window=timeline_data["df_window"],
    svg_width_m=room["svg_width_m"],
    svg_height_m=room["svg_height_m"],
    machine_radius_m=0.30,
)

persistent_machine_entities = machine_state["persistent_machine_entities"]
machine_entity_map = machine_state["machine_entity_map"]
machine_temp_lookup = machine_state["machine_temp_lookup"]

print("Persistent machines created:", len(persistent_machine_entities))

Persistent machines created: 11


initial fields and tau field

In [10]:
thermal_grid, sock_tracer, u_vel, v_vel, p = build_initial_fields(
    grid_h=room["grid_h"],
    grid_w=room["grid_w"],
    is_obstacle=room["is_obstacle"],
    T_ref=cfg.T_ref,
    supply_temp=cfg.supply_temp,
    src_y0=room["src_y0"],
    src_y1=room["src_y1"],
)

dt = compute_time_step(room["dx"], cfg.sock_speed, cfg.alpha_heat, cfg.nu_eff)

tau_field = compute_tau_field(
    (room["grid_h"], room["grid_w"]),
    room["src_y0"],
    room["src_y1"],
    room["is_obstacle"],
    base_entities_list + persistent_machine_entities,
    cfg.res,
    tau_min=100,
    tau_max=400,
)

substeps_per_frame = int(save_interval_s / dt)

print("dt =", dt)
print("frames =", cfg.frames)
print("save interval (s) =", save_interval_s)
print("substeps_per_frame =", substeps_per_frame)
print("total simulated time represented =", cfg.frames * substeps_per_frame * dt)

dt = 0.06666666666666667
frames = 120
save interval (s) = 30.0
substeps_per_frame = 450
total simulated time represented = 3600.0


step function

In [11]:
sim_time_state = {"t": 0.0}
dynamic_entities_state = {"humans": []}

def step_fn(T, tracer, u, v, p):
    update_machine_entities_for_time(
        sim_time_s=sim_time_state["t"],
        first_ts=timeline_data["first_ts"],
        unique_times=timeline_data["unique_times"],
        machine_entity_map=machine_entity_map,
        machine_temp_lookup=machine_temp_lookup,
    )

    human_entities = build_human_entities_for_time(
        sim_time_s=sim_time_state["t"],
        first_ts=timeline_data["first_ts"],
        unique_times=timeline_data["unique_times"],
        rows_by_time=timeline_data["rows_by_time"],
        svg_width_m=room["svg_width_m"],
        svg_height_m=room["svg_height_m"],
    )

    dynamic_entities_state["humans"] = human_entities
    entities_list = base_entities_list + persistent_machine_entities + human_entities

    out = cfd_step(
        T, tracer, u, v, p,
        dx=room["dx"],
        dt=dt,
        entities_list=entities_list,
        is_obstacle=room["is_obstacle"],
        res=cfg.res,
        src_y0=room["src_y0"],
        src_y1=room["src_y1"],
        hs_x=room["hs_x"],
        hs_y=room["hs_y"],
        bubble_r=cfg.bubble_r,
        alpha_heat=cfg.alpha_heat,
        nu_eff=cfg.nu_eff,
        rho=cfg.rho,
        beta_b=cfg.beta_b,
        g=cfg.g,
        T_ref=cfg.T_ref,
        sock_speed=cfg.sock_speed,
        supply_temp=cfg.supply_temp,
        smoke_diff=cfg.smoke_diff,
        pressure_iters=cfg.pressure_iters,
        max_speed=cfg.max_speed,
        tau_field=tau_field,
    )

    sim_time_state["t"] += dt
    return out

run analysis

In [12]:
thermal_grid, sock_tracer, u_vel, v_vel, p = build_initial_fields(
    grid_h=room["grid_h"],
    grid_w=room["grid_w"],
    is_obstacle=room["is_obstacle"],
    T_ref=cfg.T_ref,
    supply_temp=cfg.supply_temp,
    src_y0=room["src_y0"],
    src_y1=room["src_y1"],
)

sim_time_state = {"t": 0.0}
dynamic_entities_state = {"humans": []}

df_metrics, final_state = run_simulation_and_collect(
    thermal_grid, sock_tracer, u_vel, v_vel, p,
    step_fn=step_fn,
    frames=cfg.frames,
    substeps_per_frame=substeps_per_frame,
    dt=dt,
    is_obstacle=room["is_obstacle"],
    machine_entities=persistent_machine_entities,
    res=cfg.res,
    real_start_timestamp=timeline_data["first_ts"],
    use_circular_machine_metrics=True,
    default_machine_radius_m=0.30,
    surround_outer_radius_m=0.60,
)

csv_path = save_metric_plots(df_metrics, results_dir)

print("Saved results to:", results_dir)
print("Metrics CSV:", csv_path)
df_metrics.head()

Saved results to: ../results
Metrics CSV: ../results\metrics.csv


,room_avg_temp,room_max_temp,room_min_temp,room_std_temp,room_avg_speed,room_max_speed,machine_dbscan_cableduct_0000_box_temp,machine_dbscan_cableduct_0000_surround_temp,machine_dbscan_cableduct_0001_box_temp,machine_dbscan_cableduct_0001_surround_temp,...,machine_dbscan_screen_0000_box_temp,machine_dbscan_screen_0000_surround_temp,machine_dbscan_screen_0001_box_temp,machine_dbscan_screen_0001_surround_temp,machine_dbscan_window_0000_box_temp,machine_dbscan_window_0000_surround_temp,frame,sim_time_s,sim_time_min,real_datetime
0,20.530544,27.347015,19.867610,0.389704,0.0,0.0,20.612272,20.614678,20.578897,20.591808,...,20.597288,22.029318,20.962435,22.305399,24.384506,20.785775,1,30.0,0.5,2025-04-23 13:29:01
1,20.324629,26.162569,19.644562,0.466837,0.0,0.0,22.179412,20.521807,21.732744,20.549404,...,21.844078,21.668125,23.175073,21.890549,24.132983,20.702159,2,60.0,1.0,2025-04-23 13:29:31
2,20.144009,26.837824,19.479543,0.527032,0.0,0.0,23.037882,20.453863,21.357101,20.393299,...,21.412040,21.331784,23.072994,22.712611,24.750211,20.643902,3,90.0,1.5,2025-04-23 13:30:01
3,19.978459,25.748163,19.357166,0.460351,0.0,0.0,22.140918,20.334308,21.363636,20.263219,...,21.458814,21.003254,22.495757,22.190140,23.158482,20.516262,4,120.0,2.0,2025-04-23 13:30:31
4,19.847733,24.741993,19.266244,0.458405,0.0,0.0,22.181366,20.240046,21.608670,20.156654,...,21.767023,20.786948,22.662199,21.773499,23.553692,20.440107,5,150.0,2.5,2025-04-23 13:31:01


save gif

In [13]:
thermal_grid, sock_tracer, u_vel, v_vel, p = build_initial_fields(
    grid_h=room["grid_h"],
    grid_w=room["grid_w"],
    is_obstacle=room["is_obstacle"],
    T_ref=cfg.T_ref,
    supply_temp=cfg.supply_temp,
    src_y0=room["src_y0"],
    src_y1=room["src_y1"],
)

sim_time_state = {"t": 0.0}
dynamic_entities_state = {"humans": []}

gif_path = os.path.join(results_dir, "simulation.gif")

anim = animate_simulation(
    thermal_grid, sock_tracer, u_vel, v_vel, p,
    step_fn=step_fn,
    frames=cfg.frames,
    substeps_per_frame=substeps_per_frame,
    svg_width_m=room["svg_width_m"],
    svg_height_m=room["svg_height_m"],
    is_obstacle=room["is_obstacle"],
    y_sock_m=cfg.y_sock_m,
    sock_thickness_m=cfg.sock_thickness_m,
    hs_x_m=cfg.hotspot_x_m,
    hs_y_m=cfg.hotspot_y_m,
    v_sock_target=cfg.sock_speed,
    T_supply=cfg.supply_temp,
    dt=dt,
    entities_list=base_entities_list + persistent_machine_entities,
    real_start_timestamp=timeline_data["first_ts"],
    dynamic_entities_state=dynamic_entities_state,
    save_gif_path=gif_path,
    gif_fps=8,
    show_inline=False,
)

print("Saved GIF to:", gif_path)

Saved GIF to: ../results\simulation.gif
